# GPR68 Day 2 — RDKit processing
Reproducible notebook following the supplied Methods workflow. The only adaptation is that this SDF stores IDs under `Catalog ID` while `_Name` is blank.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/GPR68_screening')
REFERENCE = PROJECT / '01_reference_ligands'
LIBRARY = PROJECT / '02_screening_library'
PROTEIN = PROJECT / '03_protein_structure'
RDKIT_OUT = PROJECT / '04_rdkit_processing'
RDKIT_OUT.mkdir(exist_ok=True)
print('Project folder:', PROJECT)

In [ ]:
!pip -q install rdkit pandas numpy matplotlib
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, Lipinski
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print('RDKit imported successfully.')

In [ ]:
sdf_file = LIBRARY / 'MCE_5K_raw.sdf'
supplier = Chem.SDMolSupplier(str(sdf_file), removeHs=False)
mols=[]
failed=0
for mol in supplier:
    if mol is None:
        failed += 1
    else:
        mols.append(mol)
print('Successfully loaded:', len(mols))
print('Failed molecules:', failed)

In [ ]:
mol=mols[0]
print('Molecule name:', mol.GetProp('_Name') if mol.HasProp('_Name') else 'No name')
print('Available SDF properties:')
print(list(mol.GetPropNames()))

In [ ]:
records=[]
for i,mol in enumerate(mols):
    name = mol.GetProp('_Name').strip() if mol.HasProp('_Name') else ''
    if not name and mol.HasProp('Catalog ID'):
        name = mol.GetProp('Catalog ID').strip()
    if not name:
        name=f'compound_{i+1}'
    records.append({
        'compound_id': name,
        'smiles': Chem.MolToSmiles(mol),
        'MW': Descriptors.MolWt(mol),
        'LogP': Crippen.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'HBD': Lipinski.NumHDonors(mol),
        'HBA': Lipinski.NumHAcceptors(mol),
        'RotatableBonds': Lipinski.NumRotatableBonds(mol)
    })
df=pd.DataFrame(records)
print(df.shape)
df.head()

In [ ]:
output_file=RDKIT_OUT/'MCE_5K_RDKit_properties.csv'
df.to_csv(output_file,index=False)
print('Saved to:',output_file)

In [ ]:
df[['MW','LogP','TPSA']].describe()

In [ ]:
for col,title in [('MW','Molecular Weight'),('LogP','LogP'),('TPSA','TPSA')]:
    plt.figure(figsize=(8,5))
    plt.hist(df[col],bins=30)
    plt.xlabel(title)
    plt.ylabel('Number of Compounds')
    plt.title(f'MCE Library {title} Distribution')
    plt.show()